<a href="https://colab.research.google.com/github/Vinay21rout/PPE-Detection-System/blob/main/ppe_detection_model_testing__validation_14may2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
!ls "/content/drive/MyDrive/YOLOv8_model_training/"

construction_safety_yolov8.zip	models	scripts     yolov8n.pt
datasets			runs	yolo26n.pt


In [4]:
!cd "/content/drive/MyDrive/YOLOv8_model_training/models/"

In [5]:
!ls "/content/drive/MyDrive/YOLOv8_model_training/models/"

helmet-vest-no-helmet-no-vest-person


In [6]:
!ls "/content/drive/MyDrive/YOLOv8_model_training/models/helmet-vest-no-helmet-no-vest-person/"

args.yaml			 results.csv	      val_batch0_labels.jpg
BoxF1_curve.png			 results.png	      val_batch0_pred.jpg
BoxP_curve.png			 train_batch0.jpg     val_batch1_labels.jpg
BoxPR_curve.png			 train_batch1.jpg     val_batch1_pred.jpg
BoxR_curve.png			 train_batch2120.jpg  val_batch2_labels.jpg
confusion_matrix_normalized.png  train_batch2121.jpg  val_batch2_pred.jpg
confusion_matrix.png		 train_batch2122.jpg  weights
labels.jpg			 train_batch2.jpg


In [7]:
!cd "/content/drive/MyDrive/YOLOv8_model_training/models/helmet-vest-no-helmet-no-vest-person/weights"

In [9]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 19.6 MB/s eta 0:00:00


In [10]:
from ultralytics import YOLO

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [14]:
!pwd

/content


In [15]:
model=YOLO("/content/drive/MyDrive/YOLOv8_model_training/models/helmet-vest-no-helmet-no-vest-person/weights/best.pt")

In [16]:
result=model("/content/ppe_detection_demo_video.mp4",save=True)


WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

video 1/1 (frame 1/192) /content/ppe_detection_demo_video.mp4: 384x640 9 helmets, 10 persons, 8 vests, 382.5ms
video 1/1 (frame 2/192) /content/ppe_detection_demo_video.mp4: 384x640 9 helmets, 9 persons, 8 vests, 167.4ms
video 1/1 (frame 3/192) /content/ppe_detection_demo_video.mp4: 384x640 10 helmets, 10 persons, 8 vests, 150.6ms
video 1/1 (frame 4/192) /content/ppe_detection_demo_video.mp4: 384x640 9 helmets, 10 persons, 8 vests, 138.9ms
video 1/1 (fra

# Let's Use This Model in REAL TIME like live video

In [17]:
!pip install opencv-python

In [19]:
import cv2

capture=cv2.VideoCapture(0)

while True:
  ret,frame=capture.read()
  if ret==True:
    result=model(frame)
    annotated_frame=result[0].plot()
    cv2.imshow("PPE_detection",annotated_frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
      break
cap.release()
cap.destroyAllWindows()

KeyboardInterrupt: 

In [20]:
!pip install gradio

In [22]:
!mkdir /content/output

In [26]:
from ultralytics import YOLO
import gradio as gr
import cv2
import os
from PIL import Image

# Load model
model = YOLO("/content/drive/MyDrive/YOLOv8_model_training/models/helmet-vest-no-helmet-no-vest-person/weights/best.pt")

# Output folder
os.makedirs("outputs", exist_ok=True)

# IMAGE DETECTION
def detect_image(image):

    results = model(image)

    annotated_frame = results[0].plot()

    output_path = "/content/output/output_image.jpg"

    cv2.imwrite(output_path, annotated_frame)

    return output_path


# VIDEO DETECTION
def detect_video(video_path):

    cap = cv2.VideoCapture(video_path)

    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = int(cap.get(cv2.CAP_PROP_FPS))

    output_path = "outputs/output_video.mp4"

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')

    out = cv2.VideoWriter(
        output_path,
        fourcc,
        fps,
        (width, height)
    )

    while True:

        ret, frame = cap.read()

        if not ret:
            break

        results = model(frame)

        annotated_frame = results[0].plot()

        out.write(annotated_frame)

    cap.release()
    out.release()

    return output_path


# GRADIO UI
with gr.Blocks() as demo:

    gr.Markdown("# PPE Detection System")

    with gr.Tab("Image Detection"):

        image_input = gr.Image(type="pil")
        image_output = gr.Image()

        image_button = gr.Button("Detect PPE")

        image_button.click(
            fn=detect_image,
            inputs=image_input,
            outputs=image_output
        )

    with gr.Tab("Video Detection"):

        video_input = gr.Video()
        video_output = gr.Video()

        video_button = gr.Button("Detect PPE")

        video_button.click(
            fn=detect_video,
            inputs=video_input,
            outputs=video_output
        )

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://09753d647f1794b671.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
